# Assignment 1 — Part A: REST API Data Collection

## Project: VIX vs. S&P 500

**Objective:** Collect historical VIX and S&P 500 data
using the FRED REST API, clean the data, and prepare
it for analysis.

**Data source:** Federal Reserve Bank of St. Louis (FRED)

**Series:**
- VIXCLS: Cboe Volatility Index
- SP500: S&P 500 Index

## Question 1

In [2]:
# Import libraries for HTTP requests, data manipulation,
# and secure API key input.
import requests
import pandas as pd
from getpass import getpass

In [4]:
# Prompt for the FRED API key without displaying or hardcoding it.
api_key = getpass("Enter your FRED API key: ")

Enter your FRED API key:  ········


In [5]:
# Define the FRED endpoint for retrieving historical observations.
url = "https://api.stlouisfed.org/fred/series/observations"

# Specify the VIX series, JSON response format, and date range.
params = {
    "series_id": "VIXCLS",
    "api_key": api_key,
    "file_type": "json",
    "observation_start": "2026-01-01",
    "observation_end": "2026-06-30"
}

In [6]:
# Send an HTTP GET request to FRED with a 30-second timeout.
response = requests.get(url, params=params, timeout=30)

# Check the HTTP status code (200 indicates a successful response).
print(response.status_code)

# Stop execution if the API request was unsuccessful.
response.raise_for_status()

200


In [7]:
# Parse the JSON response into a Python dictionary.
data = response.json()

In [8]:
# Inspect the available keys to identify where the observations are stored.
print(data.keys())

dict_keys(['realtime_start', 'realtime_end', 'observation_start', 'observation_end', 'units', 'output_type', 'file_type', 'order_by', 'sort_order', 'count', 'offset', 'limit', 'observations'])


In [9]:
# Confirm that the API returned at least 50 raw records.
print("Number of records:", len(data["observations"]))

# Preview the first three raw observations before cleaning.
data["observations"][:3]

Number of records: 129


[{'realtime_start': '2026-10-18',
  'realtime_end': '2026-10-18',
  'date': '2026-01-01',
  'value': '.'},
 {'realtime_start': '2026-10-18',
  'realtime_end': '2026-10-18',
  'date': '2026-01-02',
  'value': '14.51'},
 {'realtime_start': '2026-10-18',
  'realtime_end': '2026-10-18',
  'date': '2026-01-05',
  'value': '14.9'}]

In [10]:
# Convert the list of JSON observations into a pandas DataFrame.
df_vix = pd.DataFrame(data["observations"])

# Display the first five rows to inspect the dataset's structure.
df_vix.head()

,realtime_start,realtime_end,date,value
0,2026-10-18,2026-10-18,2026-01-01,.
1,2026-10-18,2026-10-18,2026-01-02,14.51
2,2026-10-18,2026-10-18,2026-01-05,14.9
3,2026-10-18,2026-10-18,2026-01-06,14.75
4,2026-10-18,2026-10-18,2026-01-07,15.38


In [11]:
# Check the number of rows and columns before cleaning.
df_vix.shape

(129, 4)

In [12]:
# Inspect the original data types to identify necessary conversions.
df_vix.dtypes

realtime_start    str
realtime_end      str
date              str
value             str
dtype: object

In [13]:
# Convert VIX values from strings to numeric values.
# Non-numeric entries, such as "." for missing market observations,
# are converted to NaN.
df_vix["value"] = pd.to_numeric(df_vix["value"], errors="coerce")

In [14]:
# Count missing values in each column before removing incomplete records.
df_vix.isna().sum()

realtime_start    0
realtime_end      0
date              0
value             2
dtype: int64

In [15]:
# Remove rows without a valid VIX closing value.
# Missing market observations should not be treated as zero volatility.
df_vix = df_vix.dropna(subset=["value"])

In [16]:
# Convert observation dates from strings to datetime objects.
# This will allow us to align VIX and S&P 500 data by trading date.
df_vix["date"] = pd.to_datetime(df_vix["date"])

In [17]:
# Verify the final dimensions, data types, and remaining missing values.
print(df_vix.shape)
print(df_vix.dtypes)
print(df_vix.isna().sum())

# Preview the cleaned VIX dataset.
df_vix.head()

(127, 4)
realtime_start               str
realtime_end                 str
date              datetime64[us]
value                    float64
dtype: object
realtime_start    0
realtime_end      0
date              0
value             0
dtype: int64


,realtime_start,realtime_end,date,value
1,2026-10-18,2026-10-18,2026-01-02,14.51
2,2026-10-18,2026-10-18,2026-01-05,14.90
3,2026-10-18,2026-10-18,2026-01-06,14.75
4,2026-10-18,2026-10-18,2026-01-07,15.38
5,2026-10-18,2026-10-18,2026-01-08,15.45


### Data Cleaning Summary

The FRED API returned 129 raw VIX observations for the selected date range. After converting the JSON response into a pandas DataFrame, I inspected the dataset for missing values and incorrect data types.

The VIX closing values were initially stored as strings, and unavailable observations were represented by `"."`. I converted the values to numeric format, replaced invalid entries with `NaN`, and removed observations without a valid closing value. I also converted the date column to datetime format.

The cleaned dataset can now be used for further analysis and comparison with S&P 500 data.

## Retrieve S&P 500 Data

This section retrieves historical S&P 500 closing values using
the FRED REST API. The same date range is used as the VIX dataset
to allow both series to be compared by trading date.

In [18]:
# Define the parameters for the S&P 500 historical data request.
params_sp500 = {
    "series_id": "SP500",
    "api_key": api_key,
    "file_type": "json",
    "observation_start": "2026-01-01",
    "observation_end": "2026-06-30"
}

# Send an HTTP GET request using the same FRED API endpoint.
response_sp500 = requests.get(url, params=params_sp500, timeout=30)

# Check whether the API request was successful.
print("HTTP status:", response_sp500.status_code)

# Stop execution if the request was unsuccessful.
response_sp500.raise_for_status()

HTTP status: 200


In [19]:
# Parse the S&P 500 JSON response.
data_sp500 = response_sp500.json()

# Count the number of raw observations returned.
print("Number of S&P 500 records:", len(data_sp500["observations"]))

# Preview the first three observations.
data_sp500["observations"][:3]

Number of S&P 500 records: 129


[{'realtime_start': '2026-10-18',
  'realtime_end': '2026-10-18',
  'date': '2026-01-01',
  'value': '.'},
 {'realtime_start': '2026-10-18',
  'realtime_end': '2026-10-18',
  'date': '2026-01-02',
  'value': '6858.47'},
 {'realtime_start': '2026-10-18',
  'realtime_end': '2026-10-18',
  'date': '2026-01-05',
  'value': '6902.05'}]

In [20]:
# Convert the S&P 500 observations into a pandas DataFrame.
df_sp500 = pd.DataFrame(data_sp500["observations"])

# Preview the first five rows.
df_sp500.head()

,realtime_start,realtime_end,date,value
0,2026-10-18,2026-10-18,2026-01-01,.
1,2026-10-18,2026-10-18,2026-01-02,6858.47
2,2026-10-18,2026-10-18,2026-01-05,6902.05
3,2026-10-18,2026-10-18,2026-01-06,6944.82
4,2026-10-18,2026-10-18,2026-01-07,6920.93


In [21]:
# Inspect the data types before cleaning.
df_sp500.dtypes

realtime_start    str
realtime_end      str
date              str
value             str
dtype: object

In [22]:
# Convert S&P 500 closing values from strings to numeric values.
# Non-numeric entries, such as "." for missing market observations,
# are converted to NaN.
df_sp500["value"] = pd.to_numeric(df_sp500["value"], errors="coerce")

In [23]:
# Count missing values in each column before removing incomplete records.
df_sp500.isna().sum()

realtime_start    0
realtime_end      0
date              0
value             6
dtype: int64

In [24]:
# Remove rows without a valid S&P 500 closing value.
# Missing market observations should not be treated as an index value of zero.
df_sp500 = df_sp500.dropna(subset=["value"])

In [25]:
# Convert observation dates from strings to datetime objects.
# This will allow us to align VIX and S&P 500 data by trading date.
df_sp500["date"] = pd.to_datetime(df_sp500["date"])

In [26]:
# Verify the dimensions, data types, and missing values.
print(df_sp500.shape)
print(df_sp500.dtypes)
print(df_sp500.isna().sum())

# Preview the cleaned S&P 500 dataset.
df_sp500.head()

(123, 4)
realtime_start               str
realtime_end                 str
date              datetime64[us]
value                    float64
dtype: object
realtime_start    0
realtime_end      0
date              0
value             0
dtype: int64


,realtime_start,realtime_end,date,value
1,2026-10-18,2026-10-18,2026-01-02,6858.47
2,2026-10-18,2026-10-18,2026-01-05,6902.05
3,2026-10-18,2026-10-18,2026-01-06,6944.82
4,2026-10-18,2026-10-18,2026-01-07,6920.93
5,2026-10-18,2026-10-18,2026-01-08,6921.46


## Merge VIX and S&P 500 Data

In [27]:
# Keep only the date and closing value columns.
# Rename the values to distinguish the two financial indices.

vix = df_vix[["date", "value"]].rename(
    columns={"value": "vix_close"}
)

sp500 = df_sp500[["date", "value"]].rename(
    columns={"value": "sp500_close"}
)

In [28]:
# Merge the two datasets using their shared trading dates.
# An inner join keeps only dates present in both datasets.

df_merged = pd.merge(
    vix,
    sp500,
    on="date",
    how="inner"
)

# Preview the combined dataset.
df_merged.head()

,date,vix_close,sp500_close
0,2026-01-02,14.51,6858.47
1,2026-01-05,14.90,6902.05
2,2026-01-06,14.75,6944.82
3,2026-01-07,15.38,6920.93
4,2026-01-08,15.45,6921.46


In [29]:
# Verify the dimensions and missing values of the merged dataset.
print(df_merged.shape)
print(df_merged.isna().sum())

df_merged.head()

(123, 3)
date           0
vix_close      0
sp500_close    0
dtype: int64


,date,vix_close,sp500_close
0,2026-01-02,14.51,6858.47
1,2026-01-05,14.90,6902.05
2,2026-01-06,14.75,6944.82
3,2026-01-07,15.38,6920.93
4,2026-01-08,15.45,6921.46


### Conclusion — REST API Data Collection

I successfully retrieved 129 raw observations each for the VIX and S&P 500 from the FRED REST API. After converting the data into pandas DataFrames, I cleaned missing values and converted the relevant columns to appropriate data types.

The cleaned datasets contained 127 VIX observations and 123 S&P 500 observations. I then merged them by date using an inner join, producing a final dataset of 123 observations with no missing values.

This dataset is ready for further financial market analysis.

# Part A — Question 2: Web Scraping

## Data Source: Wikipedia — List of S&P 500 Companies

This section collects company information from Wikipedia's S&P 500
constituents table using Requests and BeautifulSoup.

The website's robots.txt was reviewed before scraping.

In [30]:
# Import libraries for retrieving, parsing, and organizing web data.
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [31]:
# Define the URL of the webpage containing S&P 500 company information.
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

# Identify the request as part of an educational web scraping project.
headers = {
    "User-Agent": "StudentDataEngineeringProject/1.0 (educational use)"
}

# Send an HTTP GET request to retrieve the webpage's HTML.
response = requests.get(url, headers=headers, timeout=30)

# Verify that the request was successful before processing the HTML.
print(response.status_code)
response.raise_for_status()

200


## Parse HTML Using BeautifulSoup

In [32]:
# Parse the HTML response so we can locate and extract webpage elements.
soup = BeautifulSoup(response.text, "html.parser")

# Display the webpage title to verify that the HTML was parsed.
print(soup.title.get_text(strip=True))

List of S&P 500 companies - Wikipedia


In [33]:
# Locate the S&P 500 constituents table using its HTML ID.
table = soup.find("table", id="constituents")

# Check whether the table was found.
print(table is not None)

True


In [34]:
# Find the header cells in the S&P 500 constituents table.
headers_html = table.find_all("th")

# Extract the text from each header cell.
column_names = [header.get_text(strip=True) for header in headers_html]

# Display the column names.
print(column_names)

['Symbol', 'Security', 'GICSSector', 'GICS Sub-Industry', 'Headquarters Location', 'Date added', 'CIK', 'Founded']


In [35]:
# Find all rows within the S&P 500 constituents table.
rows = table.find_all("tr")

# Count the number of rows found.
print("Total HTML rows:", len(rows))

Total HTML rows: 504


In [36]:
# Select the first company row, skipping the table header.
first_row = rows[1]

# Find all data cells within that row.
cells = first_row.find_all("td")

# Extract the text from each cell.
first_company = [cell.get_text(" ", strip=True) for cell in cells]

# Display the first company's information.
print(first_company)

['MMM', '3M', 'Industrials', 'Industrial Conglomerates', 'Saint Paul, Minnesota', '1957-03-04', '0000066740', '1902']


## Extract Company Records

In [37]:

# Create empty lists to store company records and hyperlinks.
companies = []
company_links = []

# Loop through all company rows, skipping the header row.
for row in rows[1:]:

    # Find the data cells within each company row.
    cells = row.find_all("td")

    # Extract the text from each cell.
    company = [cell.get_text(" ", strip=True) for cell in cells]

    # Find the HTML link element in the company name cell.
    link_element = cells[1].find("a")

    # Extract the href attribute (company's Wikipedia link).
    # If no link exists, store None instead.
    company_link = link_element.get("href") if link_element else None

    # Add the company's information to our list.
    companies.append(company)

    # Store the extracted hyperlink separately.
    company_links.append(company_link)

# Verify the number of company records extracted.
print("Total companies extracted:", len(companies))

Total companies extracted: 503


In [38]:

# Convert the extracted company records into a pandas DataFrame.
df_companies = pd.DataFrame(companies, columns=column_names)

# Add the extracted company hyperlinks as a new column.
df_companies["company_link"] = company_links

# Preview the first five records.
df_companies.head()


,Symbol,Security,GICSSector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded,company_link
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,0000066740,1902,https://en.wikipedia.org/wiki/3M
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee , Wisconsin",2017-07-26,0000091142,1916,https://en.wikipedia.org/wiki/A._O._Smith
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,0000001800,1888,https://en.wikipedia.org/wiki/Abbott_Laboratories
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,0001551152,2013 (1888),https://en.wikipedia.org/wiki/AbbVie
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin , Ireland",2011-07-06,0001467373,1989,https://en.wikipedia.org/wiki/Accenture


## Data Cleaning and Validation

In [56]:
# Check the dimensions, column data types, and missing values.
print(df_companies.shape)
print(df_companies.dtypes)
print(df_companies.isna().sum())

(503, 8)
Symbol                   str
Security                 str
GICSSector               str
GICS Sub-Industry        str
Headquarters Location    str
Date added               str
CIK                      str
Founded                  str
dtype: object
Symbol                   0
Security                 0
GICSSector               0
GICS Sub-Industry        0
Headquarters Location    0
Date added               0
CIK                      0
Founded                  0
dtype: int64


In [40]:

# Rename columns using consistent, descriptive snake_case names.
df_companies.columns = [
    "symbol",
    "company",
    "sector",
    "sub_industry",
    "headquarters",
    "date_added",
    "cik",
    "founded",
    "company_link"
]

# Verify the updated column names.
print(df_companies.columns)


Index(['symbol', 'company', 'sector', 'sub_industry', 'headquarters',
       'date_added', 'cik', 'founded', 'company_link'],
      dtype='str')


In [41]:
# Check for empty strings that pandas may not recognize as missing values.
print((df_companies == "").sum())

symbol          0
company         0
sector          0
sub_industry    0
headquarters    0
date_added      0
cik             0
founded         0
company_link    0
dtype: int64


In [42]:
# Convert the date_added column from text to datetime.
# Invalid or unrecognized dates will become NaT (missing datetime).
df_companies["date_added"] = pd.to_datetime(
    df_companies["date_added"],
    errors="coerce"
)

# Verify the updated data type and check for invalid dates.
print(df_companies["date_added"].dtype)
print(df_companies["date_added"].isna().sum())

datetime64[us]
0


## Final Data Validation

In [43]:
# Preview the first five cleaned company records.
display(df_companies.head())

# Verify the final number of rows and columns.
print("Dataset shape:", df_companies.shape)

# Verify the data types after cleaning.
print("\nData types:")
print(df_companies.dtypes)

# Check for any remaining missing values.
print("\nMissing values:")
print(df_companies.isna().sum())

,symbol,company,sector,sub_industry,headquarters,date_added,cik,founded,company_link
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,0000066740,1902,https://en.wikipedia.org/wiki/3M
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee , Wisconsin",2017-07-26,0000091142,1916,https://en.wikipedia.org/wiki/A._O._Smith
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,0000001800,1888,https://en.wikipedia.org/wiki/Abbott_Laboratories
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,0001551152,2013 (1888),https://en.wikipedia.org/wiki/AbbVie
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin , Ireland",2011-07-06,0001467373,1989,https://en.wikipedia.org/wiki/Accenture


Dataset shape: (503, 9)

Data types:
symbol                     str
company                    str
sector                     str
sub_industry               str
headquarters               str
date_added      datetime64[us]
cik                        str
founded                    str
company_link               str
dtype: object

Missing values:
symbol          0
company         0
sector          0
sub_industry    0
headquarters    0
date_added      0
cik             0
founded         0
company_link    1
dtype: int64


### Conclusion — Web Scraping

I successfully extracted 503 company records from Wikipedia's S&P 500 constituents table using Requests and BeautifulSoup. After converting the extracted HTML data into a pandas DataFrame, I standardized the column names, checked for missing values, and converted the date column to datetime.

The final dataset contains 503 observations and 8 columns, with no missing values detected.